In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.append('/image-differential-annotator/')
# sys.path.append('/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/image-differential-annotator')

from annotator.annotation import runAnnotation, jumpStart
from annotator.combineCDF import getDiscreteCombinedCDFofAllFeatures as PCMA
from annotator.stqutils import loadAd, preparePatchesWSI, getPatchRepresentation, inferProb, showProbImg

from tqdm import tqdm
import numpy as np
import pandas as pd
import pickle

import matplotlib.pyplot as plt

qs = np.linspace(0.05, 0.95, 10, endpoint=True)

classifierPaths = 'classifiers/'
if not os.path.exists(classifierPaths):
    os.makedirs(classifierPaths)

In [ ]:
outsSTQpath = '/projects/activities/kappsen-tmc/USERS/domans/results-kidney-wedges/'
samples  = ['JAX_002_KD_C', 'JAX_003_KD_C', 'JAX_004_KD_C', 'JAX_005_KD_C', 
            'JAX_006_KD_C', 'JAX_007_KD_C', 'JAX_008_KD_C', 'JAX_009_KD_C', 
            'JAX_010_KD_C', 'JAX_011_KD_C', 'JAX_012_KD_C', 'JAX_013_KD_C', 
            'JAX_014_KD_C', 'JAX_015_KD_C', 'JAX_016_KD_C', 'JAX_017_KD_C', 'JAX_018_KD_C']

# outsSTQpath = '/projects/activities/kappsen-tmc/USERS/domans/results-kidney-001/'
# samples = ['SA_001_L_C_a', 'SA_001_L_CM_w', 'SA_001_L_M_j', 'SA_001_L_P_mnop']

# outsSTQpath = '/projects/activities/kappsen-tmc/histology/kidney/mouse-kidney/results-STQ'
# samples  = ['SN_73363_K', 'SN_73364_K', 'SN_73365_K', 'SN_73366_K', 
#             'SN_73367_K', 'SN_73368_K', 'SN_73369_K', 'SN_73370_K', 
#             'SN_73371_K', 'SN_73372_K', 'SN_73373_K', 'SN_73374_K']

In [ ]:
# If L is None, then the annotator will do lazy loading of patches with Zarr, else load entire images at requested level L
L = None
ts = 56 # Center-to-center distance between tiles (not size of a tile!)
mpp = 0.25 # Image pixel size
N = 8 # patch size, e.g., 8 by 8 tiles
fname='img.data.ctranspath-1.h5ad' # 'img.data.ctranspath-1.h5ad' or 'features/false-1-ctranspath_features.tsv.gz'

# Load the STQ data for each sample
ads = {}
imgs = {}
for sample in tqdm(samples):
    ads[sample], imgs[sample] = loadAd(f'{outsSTQpath}{sample}/', fname=fname, L=L)

# Prepare the patches coordinates for each sample and concatenate them into a single DataFrame
patchCoordinates = pd.concat([preparePatchesWSI(ads[sample].obs, N=N, spacing=ts/mpp, sample_id=sample) for sample in tqdm(samples)], axis=0) # N=8

# Get the patch SAMPLER representations for each sample and combine them into a single DataFrame
patchesCDFs = pd.concat([getPatchRepresentation(ads[sample], patchCoordinates.xs(sample, level='sample', axis=0), qs, sample_id=sample) for sample in tqdm(samples)], axis=0)

In [ ]:
startParams = {}
jumpStart(patchCoordinates, patchesCDFs, imgs, startParams, ncols=6, nrows=5, ads=ads, L=1 if L is None else L, sh=int(ts/2)/mpp, figsize=(3, 3), seed=1, maxN=10**3)

In [ ]:
clf, plog, bp = {}, [], {}
try: bp.update({startParams['selected_patch']: 'positive'})
except: pass
runAnnotation(patchCoordinates, patchesCDFs, imgs, bp, clf, plog, ads=ads, qs=qs, minN=1,
            figsize=(5, 5), augFunc=PCMA, alpha=0.5, R=1, cmapColors=['lightcoral', 'blue'], # (5, 5)
            seed=0, randomness=0.5, L=1 if L is None else L, sh=int(ts/2)/mpp)

In [ ]:
# To save the classifier
if False:
    with open(f'{classifierPaths}/clf-somename.pkl', 'wb') as tempfile:
        pickle.dump(clf['clf'], tempfile)

# To load the classifier back
if False:
    clf = {}
    with open(f'{classifierPaths}/clf-somename.pkl', 'rb') as tempfile:
        clf['clf'] = pickle.load(tempfile)

In [ ]:
# To run inference on the entire slides
for i in range(len(samples)):
    infSample  = samples[i]
    x, y, p = inferProb(ads[infSample], clf['clf'], qs, tsize=ts/mpp, R=2)
    showProbImg(x, y, p, f=1, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample)

In [ ]:
# To run inference on the entire slides and save the images, and then display them in a grid
for i in range(len(samples)):
    infSample  = samples[i]
    x, y, p = inferProb(ads[infSample], clf['clf'], qs, tsize=ts/mpp, R=2)
    showProbImg(x, y, p, f=1, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample,
                saveName=f'{infSample}.png', dpi=150)

nc = 4
nr = int(np.ceil(len(samples) / nc))
fig, axs = plt.subplots(nr, nc, figsize=(nc*3, nr*3))
axs = axs.flatten()
for i in range(nr*nc):
    ax = axs[i]
    if i < len(samples):
        img = plt.imread(f'{samples[i]}.png')
        ax.imshow(img)
    ax.axis('off')
fig.tight_layout()
plt.show()